In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

load_dotenv()

llm = ChatOpenAI(
    model="xiaomi/mimo-v2.5",
    temperature=0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

@tool
def calcular_precio(cantidad: float, precio_unitario: float) -> str:
    """Calcula el precio total de un pedido."""
    total = cantidad * precio_unitario
    return f"El precio total es: {total:.2f} EUR"

# --- AGENTE INVESTIGADOR ---
investigador = create_agent(
    model=llm,
    tools=[DuckDuckGoSearchRun()],
    system_prompt="Eres un investigador experto. Busca información precisa y actualizada.",
)


@tool
def investigar(pregunta: str) -> str:
    """Investiga un tema en internet. Input: pregunta de investigación."""
    resultado = investigador.invoke({
        "messages": [{"role": "user", "content": pregunta}]
    })
    return resultado["messages"][-1].content


# --- AGENTE ANALISTA ---
analista = create_agent(
    model=llm,
    tools=[calcular_precio],
    system_prompt="Eres un analista de datos. Extrae insights clave de la información.",
)


@tool
def analizar(datos: str) -> str:
    """Analiza datos o información. Input: datos a analizar."""
    resultado = analista.invoke({
        "messages": [{"role": "user", "content": datos}]
    })
    return resultado["messages"][-1].content

@tool
def save_to_markdown(content: str, filename: str) -> str:
    """Guarda contenido en un archivo Markdown.
    Args:
        content: El contenido a guardar.
        filename: El nombre del archivo (con extensión .md).
    """
    with open(filename, "w") as f:
        f.write(content)
    return f"Contenido guardado en {filename}"

# --- AGENTE SUPERVISOR ---
supervisor = create_agent(
    model=llm,
    tools=[investigar, analizar, save_to_markdown],
    system_prompt=(
        "Eres el supervisor de un equipo de investigación. "
        "Tienes acceso a: investigar (experto en búsqueda web) y analizar (experto en análisis de datos). "
        "Coordina el equipo para producir informes completos."
        "Cuando recibas el informe, guárdalo a markdown"
    ),
)


resultado = supervisor.invoke({
    "messages": [{"role": "user", "content": "Analiza el mercado de energías renovables en Europa en 2024"}]
})
print(resultado["messages"][-1].content)

## ✅ Informe completado y guardado

He coordinado al equipo de investigación y análisis para producir un informe ejecutivo completo sobre el **Mercado de Energías Renovables en Europa 2024**. El archivo ha sido guardado como:

📄 **`Informe_Mercado_Energias_Renovables_Europa_2024.md`**

---

### 📋 Resumen de lo que contiene el informe:

| Sección | Contenido |
|---------|-----------|
| **Resumen Ejecutivo** | 5 hallazgos clave y conclusión estratégica |
| **1. Estado General** | 848 GW instalados, 47.5% electricidad renovable, récord de inversión |
| **2. Fuentes de energía** | Solar (334 GW), Eólica (246-285 GW), Hidro (259 GW), Biomasa, Geotermia |
| **3. Ranking países** | Austria lidera (90.1%), Bélgica en cola (~15%), análisis de 3 clusters regionales |
| **4. Marco político** | Green Deal, Fit for 55, RED III, REPowerEU, CBAM |
| **5. Desafíos críticos** | Redes, almacenamiento, cadena suministro, permisos, inversión |
| **6. Oportunidades** | Ahorro 130k M€/año, hidrógeno verde, 